# DA107.4 Metrocar project final week

## Project Overview: Funnel Analysis Mastery Project

This project aims to analyze the customer funnel of Metrocar, a ride-sharing app (similar to Uber/Lyft), to identify areas for improvement and optimization. We will use SQL for data collection and Pandas for further analysis and visualisation. The stakeholders have asked several business questions that can uncover valuable insights for improving specific areas of the customer funnel. Your task is to conduct a funnel analysis and present your analysis and recommendation. Explain your reason for your recommendation based on insights retrieved from the data. 

### Project Logistics
**Day 1: Explore the Metrocar Data with SQL & Pandas:**
The first day will mainly focus on understanding the dataset using Pandas. The dataset contains many tables and properties and requires understanding of the business (ride-sharing) and the company’s funnel.

**Day 2 & 3:  Developing Metrocar Funnel Metrics:**
The second and third day will be focused on analyzing the data and reaching conclusions regarding the given business questions.
We will also start to build meaningful visualisations.

**Day 4: (Interactive) Visualisations:**
On the fourth day, you should refine your visualisations respecting Data Storytelling rules.
You can potentially dive into plotly and dash and make your funnel visualisation interactive.

**Day 5: Present the Funnel Results to Stakeholders:**
The last day will be mostly reserved for preparing and recording the final submission based on your insights and recommendations.

## Project Background

### About Metrocar:
Metrocar's business model is based on a platform that connects riders with drivers through a mobile application. Metrocar acts as an intermediary between riders and drivers, providing a user-friendly platform to connect them and facilitate the ride-hailing process.




### Metrocar’s Funnel:
**App Download:** A user downloads the Metrocar app from the App Store or Google Play Store.

**Signup:** The user creates an account in the Metrocar app, including their name, email, phone number, and payment information.

**Request Ride:** The user opens the app and requests a ride by entering their pickup location, destination, and ride capacity (2 to 6 riders).

**Driver Acceptance:** A nearby driver receives the ride request and accepts the ride.

**Ride:** The driver arrives at the pickup location, and the user gets in the car and rides to their destination.

**Payment:** After the ride, the user is charged automatically through the app, and a receipt is sent to their email.

**Review:** The user is prompted to rate their driver and leave a review of their ride experience.

Similar to other customer funnels, there will be drop-offs at every stage of the funnel, which is why funnel analysis can be helpful in identifying areas for improvement and optimization. For example, Metrocar may analyze the percentage of users who download the app but do not complete the registration process, or the percentage of users who request a ride but cancel before the driver arrives.

P.S. A **funnel** is a strategic framework used in marketing and sales to guide potential customers through a series of stages, with the aim of converting them into paying customers.

## Business questions:
You will need to analyze the data and make recommendations based on the following business questions:

- What steps of the funnel should we research and improve? Are there any specific drop-off points preventing users from completing their first ride?

- Metrocar currently supports 3 different platforms: ios, android, and web. To recommend where to focus our marketing budget for the upcoming year, what insights can we make based on the platform?

- What age groups perform best at each stage of our funnel? Which age group(s) likely contain our target customers?

- Surge pricing is the practice of increasing the price of goods or services when there is the greatest demand for them. If we want to adopt a price-surging strategy, what does the distribution of ride requests look like throughout the day?

- What part of our funnel has the lowest conversion rate? What can we do to improve this part of the funnel?

## The Dataset 
This dataset is inspired by publicly available datasets for Uber/Lyft. The data for this dataset was generated specifically for this project.

Connect to the Metrocar database:
Set `METROCAR_DATABASE_URL` in your local environment before running this notebook:

[Database connection details omitted from this public copy.]

### Getting the Data into IDE:

In [40]:
import os
import pandas as pd
import sqlalchemy as sa

# connect to the database using the credentials provided by Neon Tech
engine = sa.create_engine(os.environ["METROCAR_DATABASE_URL"])

# Check if the engine connects successfully
try:
    connection = engine.connect().execution_options(isolation_level="AUTOCOMMIT")
    print("Connection successful!")
except sa.exc.SQLAlchemyError as e:
    print(f"Error connecting to the database: {e}")

Connection successful!


You can use the inspector to get an overview of the available tables.

In [41]:
# create an inspector object to get the table names 
inspector = sa.inspect(engine) 
table_names = inspector.get_table_names()
table_names

['transactions', 'signups', 'ride_requests', 'reviews', 'app_downloads']

In [53]:
f"SELECT * FROM {table_names[0]}"

'SELECT * FROM transactions'

In [59]:
# Ensure the connection is open before executing the query
if connection.closed:
	connection = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

play1 = pd.read_sql_query("SELECT * FROM transactions LIMIT 5", connection)
print(play1)
# .read_sql_query() function is specifically for SELECT queries and returns a DataFrame object with the results of the query. 
# .read_sql() function is more general and can be used for any SQL query. Accepts a SQL query and table names and returns a DataFrame object with the results of the query.
# Close the connection
connection.close()

   transaction_id  ride_id  purchase_amount_usd charge_status  \
0        10000000  3000000                13.55      Approved   
1        10000001  3000001                27.77      Approved   
2        10000002  3000002                21.84      Approved   
3        10000003  3000004                26.86      Approved   
4        10000004  3000005                21.72      Approved   

       transaction_ts  
0 2021-03-28 19:11:00  
1 2021-11-10 16:59:00  
2 2021-09-08 21:03:00  
3 2021-05-28 09:21:00  
4 2021-12-01 16:59:00  


In [42]:
#key - table name
#value - dataframe


tables = {} # create an empty dictionary to store the tables
for table_name in table_names: # loop through the table names
  tables[table_name] = pd.read_sql_table(table_name, connection) # read the table into a dataframe using the connection object 
tables.keys() # display the table names in the dictionary 

dict_keys(['transactions', 'signups', 'ride_requests', 'reviews', 'app_downloads'])

# Constructing the Customer Funnel:

To address the business questions, you need to write python code that combines the data from different steps of the funnel and calculate the drop-off from one step to another.

💡
It is important that we first explore the different steps and see the level of detail we have available to count in each step. We need to find a common basis for all steps of the funnel.
The counts should be strictly decreasing through our funnel, never increasing!

An example of how to choose the right level of granularity can be seen in the “Adnimate” Live Sessions and as a recap in the next chapter on the Adnimate business case.

Once we have the counts, we can display them in a proportionate manner. There is two common ways to show the percent-wise change in each step of the tunnel:

- **Percent of Top:** This calculation involves measuring the conversion rate as a percentage of the total number of users who entered the top of the funnel. The top of the funnel represents the initial stage where users enter or show interest in a particular process or journey. For our Metrocar example, app downloads might represent the top of the funnel.

- **Percent of Previous:** This calculation involves measuring the conversion rate as a percentage of the users who proceeded to the current stage of the funnel, relative to the number of users who were at the previous stage. In other words, it tracks the progression of users through each stage of the funnel.

In addition, here are some references for specific Pandas functions that will be useful for the project:

📚 Useful Pandas functions
- Aggregate methods:
1. .count()
2. .sum()
- Methods to refer to other rows:
1. .iloc[]
2. .shift()

## Tips & Tricks 💡
### User-Level Granularity
In order to analyze the funnel at a user-level, it’s important to maintain view each dataframe at a user-level granularity. As an example, since the signups tables is already at a user-level granularity (user_id is the primary key) then it is guaranteed that a user_id only ever appears once in the user table. 

In other words, the # of distinct users is exactly the same as total users in the table:

In [43]:
# display the table names in the dictionary 
print(tables.keys()) 
# display the first 5 rows of the signups table
tables['signups'].head(2)
# what is the number of sum users and nunique users in the signups table?
tables['signups']['user_id'].agg(["count","nunique"]) 

dict_keys(['transactions', 'signups', 'ride_requests', 'reviews', 'app_downloads'])


count      17623
nunique    17623
Name: user_id, dtype: int64

However, ride_requests is at the ride_id granularity which means that the same user_id can appear on multiple rows. In other words, one user can request multiple rides:

In [44]:
tables['ride_requests']['user_id'].agg(["count","nunique"])

count      385477
nunique     12406
Name: user_id, dtype: int64

## In a user-level funnel analysis, we would want to know:
1.  **The number of users that downloaded the app** 
2.  **How many users signed up for an account**
3.  **How many users requested a ride**
4.  **How many users completed a ride**

### **1. The number of users that downloaded the app & 2. Signed up for an account**

In [45]:
tables.keys()

dict_keys(['transactions', 'signups', 'ride_requests', 'reviews', 'app_downloads'])

In [46]:
tables['app_downloads'].head(2)

,app_download_key,platform,download_ts
0,06f49bcc6895f888eba41043f95348ba,android,2021-05-13 13:12:06
1,60d79d5ac63159a5dffc13e42d87e070,android,2021-01-17 17:40:24


In [47]:
''' SQL Quiz: 1. How many times was the app downloaded? '''
# 23608 times the app was downloaded
tables['app_downloads']['download_ts'].count()

np.int64(23608)

In [48]:
''' SQL Quiz: 2. How many users signed up on the app? '''
# 17623 users signed up on the app 
tables['signups']['signup_ts'].count()  

np.int64(17623)

In [49]:
downloads_signups = pd.merge(tables['app_downloads'], tables['signups']
                             , left_on='app_download_key'
                             , right_on='session_id'
                             , how='left')
downloads_signups['is_downloaded'] = downloads_signups['app_download_key'].notnull()
downloads_signups['is_signedup'] = downloads_signups['signup_ts'].notnull()

print(downloads_signups.shape)
print(downloads_signups.head(2))

(23608, 9)
                   app_download_key platform         download_ts   user_id  \
0  06f49bcc6895f888eba41043f95348ba  android 2021-05-13 13:12:06  106328.0   
1  60d79d5ac63159a5dffc13e42d87e070  android 2021-01-17 17:40:24       NaN   

                         session_id           signup_ts age_range  \
0  06f49bcc6895f888eba41043f95348ba 2021-05-14 05:05:06   Unknown   
1                               NaN                 NaT       NaN   

   is_downloaded  is_signedup  
0           True         True  
1           True        False  


In [52]:
'''Full_funnel Question: 1. The number of users that downloaded the app'''
''' SQL Quiz: 1. How many times was the app downloaded? '''
# 23608 times app is downloaded and that is the nuber of users that downloaded the app
# P.S. in all tables/columns :)
print(tables['app_downloads']['download_ts'].count()) # 23608

print(downloads_signups['download_ts'].count()) # 23608
print(downloads_signups['is_downloaded'].sum()) # 23608

# Using SQL to count the number of app downloads
query = "SELECT COUNT(*) FROM app_downloads"
with engine.connect() as connection:
	result = connection.execute(query).fetchone()
	print(result[0])  # 23608

23608
23608
23608


ObjectNotExecutableError: Not an executable object: 'SELECT COUNT(*) FROM app_downloads'

In [ ]:
'''Full_funnel Question: 2. How many users signed up for an account'''
''' SQL Quiz: 2. How many users signed up on the app? '''
# 17623 users signed up on the app in all tables/columns :)
print(tables['signups']['signup_ts'].count()) # 17623
print(downloads_signups['signup_ts'].count()) # 17623
print(downloads_signups['is_signedup'].sum()) # 17623

17623
17623
17623


In [ ]:
print(downloads_signups.head(2))

                   app_download_key platform         download_ts   user_id  \
0  06f49bcc6895f888eba41043f95348ba  android 2021-05-13 13:12:06  106328.0   
1  60d79d5ac63159a5dffc13e42d87e070  android 2021-01-17 17:40:24       NaN   

                         session_id           signup_ts age_range  \
0  06f49bcc6895f888eba41043f95348ba 2021-05-14 05:05:06   Unknown   
1                               NaN                 NaT       NaN   

   is_downloaded  is_signedup  
0           True         True  
1           True        False  


In [ ]:
# copy downloads_signups to a new dataframe called funnel_1st_half
funnel_1st_half = downloads_signups.copy()
print(funnel_1st_half.shape)

(23608, 9)


### 3.  **How many users requested a ride** & 4.  **How many users completed a ride**
Notice that for #3 and and #4 we aren’t concerned with the number of rides completed per user. We are only concerned whether or not a user requested a ride (at least one) and completed a ride (at least one). Here is one way to answer 3 and 4 (similar to SQL):


#### 3&4. Step_1: First we generate a new column called 'is_finished'
We can check check if there was a dropoff (signified by a non-null time_stamp). 

***We can get this bool out of .notna()***

From booleans we can get an answer to both the 3rd and 4th funnel questions:

3.  How many users requested a ride = 'is_finished'.count()
4.  How many users completed a ride = 'is_finished'.sum()




In [ ]:
print(tables.keys())
print(tables['ride_requests'].shape)

dict_keys(['transactions', 'signups', 'ride_requests', 'reviews', 'app_downloads'])
(385477, 10)


In [ ]:
# create a new column called is_finished in the ride_requests table
tables['ride_requests']['is_finished'] = tables['ride_requests']['dropoff_ts'].notna()
print(tables['ride_requests'].head(2))

# display the first 5 rows of the ride_requests table with the new column 
print(tables['ride_requests'].loc[:5,['ride_id', 'user_id', 'is_finished']])
print(tables['ride_requests'].shape) # (385477, 11)

   ride_id  user_id  driver_id          request_ts accept_ts  \
0  3165680   113114        NaN 2021-10-20 20:06:00       NaT   
1  3166254   104347        NaN 2021-04-30 19:30:00       NaT   

            pickup_location          dropoff_location pickup_ts dropoff_ts  \
0  40.89412142 -73.93941019   40.89125781 -73.9195172       NaT        NaT   
1  40.90122311 -73.94995182  40.69902951 -74.02289423       NaT        NaT   

            cancel_ts  is_finished  
0 2021-10-20 20:26:00        False  
1 2021-04-30 19:49:00        False  
   ride_id  user_id  is_finished
0  3165680   113114        False
1  3166254   104347        False
2  3166727   103932        False
3  3167197   108224        False
4  3167351   108690        False
5  3167502   111387        False
(385477, 11)


#### Step_2: Here I'll create new dataframe 'ride_requests_df' to avoid changing the original dictionary
All unnecessary columns will be removed and the is_finished column will be converted to 2 columns:

is_requsted, is_completed

In [ ]:
''' copy the ride_requests table to a new dataframe to avoid changing the original tables '''
ride_requests = tables['ride_requests'].copy()
# copy is_finished column to a new column called is_requested (later booleans will be counted)
ride_requests['is_requested'] = ride_requests['is_finished']

# rename is_requested column to is_completed (later booleans will be summed)
ride_requests.rename(columns={'is_finished':'is_completed'}, inplace=True)  

# leave only the columns user_id, is_requested, is_completed
ride_requests = ride_requests[['ride_id', 'user_id', 'is_requested', 'is_completed']]

print(ride_requests.shape) # (385477, 4)
print(ride_requests.head(2)) # display the first 2 rows of the ride_requests dataframe    

(385477, 4)
   ride_id  user_id  is_requested  is_completed
0  3165680   113114         False         False
1  3166254   104347         False         False


#### 3&4. b) Now we can group by user_id and guarantee that each user will only appear once in this table:

To aggregate, we may use the .any() method and figure out users that have finished at least of their booked ride:

In [ ]:
'''The .any() function is used to check if any element in the group is True. 
( True if the user has finished a ride, False if the user has not finished a ride)'''
# as_index = False - Look at Live Session - Krystyna Mon Oct 14 2024 Rec on 00:21:45 
funnel_2nd_half = ride_requests.groupby("user_id", as_index = False).agg({ 
    "is_requested": "any", 
    "is_completed": "any" })
print(funnel_2nd_half.shape) # 12406 rows, 3 columns

# count the is_requested column to get the number of users who requested a ride
total_users_ride_requested = funnel_2nd_half["is_requested"].count() # 12406
print(total_users_ride_requested)

# sum the is_completed column to get the number of users who completed a ride
total_users_ride_completed = funnel_2nd_half["is_completed"].sum() # 6233
print(total_users_ride_completed)

print(funnel_2nd_half.head(2)) # display the first 2 rows of the funnel_2nd_half dataframe

(12406, 3)
12406
6233
   user_id  is_requested  is_completed
0   100000         False         False
1   100001         False         False


## Final Step: Completing a Full Funnel

In [ ]:
# left join on user_id column
full_funnel_v3 = pd.merge(funnel_1st_half, funnel_2nd_half, on='user_id', how='left')  
print(full_funnel_v3.shape) # (23608, 11)
print(full_funnel_v3.head(2)) # display the first 2 rows of the full_funnel_v3 dataframe

(23608, 11)
                   app_download_key platform         download_ts   user_id  \
0  06f49bcc6895f888eba41043f95348ba  android 2021-05-13 13:12:06  106328.0   
1  60d79d5ac63159a5dffc13e42d87e070  android 2021-01-17 17:40:24       NaN   

                         session_id           signup_ts age_range  \
0  06f49bcc6895f888eba41043f95348ba 2021-05-14 05:05:06   Unknown   
1                               NaN                 NaT       NaN   

   is_downloaded  is_signedup is_requested is_completed  
0           True         True          NaN          NaN  
1           True        False          NaN          NaN  


In [ ]:
print(full_funnel_v3['is_downloaded'].sum()) # 23608
print(full_funnel_v3['is_signedup'].sum()) # 17623
print(full_funnel_v3['is_requested'].count()) # 12406
print(full_funnel_v3['is_completed'].sum()) # 6233
# show unique values in the user_id column
print(full_funnel_v3['user_id'].nunique()) # 17623


23608
17623
12406
6233
17623


In [ ]:
# take only 4 last columns with .loc() function
full_funnel = full_funnel_v3.iloc[:, -4:].agg({
    'is_downloaded': 'sum'
    , 'is_signedup': 'sum'
    , 'is_requested': 'count'
    , 'is_completed': 'sum' })
print(full_funnel) # display the full_funnel dataframe
print(full_funnel.index) # display the index of the full_funnel dataframe


is_downloaded    23608
is_signedup      17623
is_requested     12406
is_completed      6233
dtype: int64
Index(['is_downloaded', 'is_signedup', 'is_requested', 'is_completed'], dtype='object')


In [ ]:
# rename the columns in the full_funnel dataframe
full_funnel_users = full_funnel.rename(index={
    'is_downloaded': 'App Downloads'
    , 'is_signedup': 'Signups'
    , 'is_requested': 'Ride Requests'
    , 'is_completed': 'Ride Completions'
})
print(full_funnel_users)

# print index in the full_funnel_users dataframe
print(full_funnel_users.index)

# print columns in the full_funnel_users dataframe
print(full_funnel_users.values)

App Downloads       23608
Signups             17623
Ride Requests       12406
Ride Completions     6233
dtype: int64
Index(['App Downloads', 'Signups', 'Ride Requests', 'Ride Completions'], dtype='object')
[23608 17623 12406  6233]


In [ ]:
from plotly.graph_objects import *
import numpy as np

In [ ]:
# copy the full_funnel_users dataframe to full_funnel
full_funnel = full_funnel_users.copy()
print(full_funnel)

App Downloads       23608
Signups             17623
Ride Requests       12406
Ride Completions     6233
dtype: int64


In [ ]:
fig = Figure(data = Funnel(x = full_funnel.values,
                           y = full_funnel.index,
                           textinfo = 'value+percent initial'))
fig.show()

In [ ]:
fig = Figure(data = Funnel(x = full_funnel.values,
                           y = full_funnel.index,
                           textinfo = "value+percent initial",
                           opacity=0.65,
                           connector = {"line": {"color": "royalblue",
                                                 "dash": "dot",
                                                 "width": 3}}))
fig.show()

In [ ]:
# rename and copy full_funnel_v3 to lead_base_table 
lead_base_table = full_funnel_v3.copy() 
print(lead_base_table.head())

                   app_download_key platform         download_ts   user_id  \
0  06f49bcc6895f888eba41043f95348ba  android 2021-05-13 13:12:06  106328.0   
1  60d79d5ac63159a5dffc13e42d87e070  android 2021-01-17 17:40:24       NaN   
2  a3e52e50d379c3da808c4d8864f0d996  android 2021-01-11 04:02:52  100497.0   
3  2f7551cdd9a0a658350394e51bc74de3  android 2021-07-22 21:00:34  109802.0   
4  81adc238826a8dce8a706c083abc095e      web 2021-04-24 01:43:54  105441.0   

                         session_id           signup_ts age_range  \
0  06f49bcc6895f888eba41043f95348ba 2021-05-14 05:05:06   Unknown   
1                               NaN                 NaT       NaN   
2  a3e52e50d379c3da808c4d8864f0d996 2021-01-12 01:58:52     45-54   
3  2f7551cdd9a0a658350394e51bc74de3 2021-07-24 02:41:34   Unknown   
4  81adc238826a8dce8a706c083abc095e 2021-04-24 01:52:54     35-44   

   is_downloaded  is_signedup is_requested is_completed  
0           True         True          NaN          NaN  


In [ ]:
funnel_steps = full_funnel.index
funnel_steps

Index(['App Downloads', 'Signups', 'Ride Requests', 'Ride Completions'], dtype='object')

In [ ]:
full_funnel_by_platform = lead_base_table.groupby('platform').agg({
    'is_downloaded': 'sum'
    , 'is_signedup': 'sum'
    , 'is_requested': 'count'
    , 'is_completed': 'sum' }).T

full_funnel_by_platform

platform,android,ios,web
is_downloaded,6935,14290,2383
is_signedup,5148,10728,1747
is_requested,3619,7550,1237
is_completed,1830,3792,611


In [ ]:
# sum the values in the full_funnel_by_age dataframe by row
# axis=1 means sum by row 
print(full_funnel_by_platform.sum(axis=1))
print(full_funnel)

is_downloaded    23608
is_signedup      17623
is_requested     12406
is_completed      6233
dtype: object
App Downloads       23608
Signups             17623
Ride Requests       12406
Ride Completions     6233
dtype: int64


In [ ]:
traces = list()
for platform in full_funnel_by_platform.columns:
  trace = Funnel(x = full_funnel_by_platform[platform].values,
                 y = full_funnel_by_platform.index,
                 textinfo = "value+percent initial",
                 name = platform)
  traces.append(trace)


fig = Figure(data = traces,
             layout = Layout(title = "Full funnel by platform"))
fig.show()

In [ ]:
full_funnel

App Downloads       23608
Signups             17623
Ride Requests       12406
Ride Completions     6233
dtype: int64

In [ ]:
lead_base_table.head()

,app_download_key,platform,download_ts,user_id,session_id,signup_ts,age_range,is_downloaded,is_signedup,is_requested,is_completed
0,06f49bcc6895f888eba41043f95348ba,android,2021-05-13 13:12:06,106328.0,06f49bcc6895f888eba41043f95348ba,2021-05-14 05:05:06,Unknown,True,True,NaN,NaN
1,60d79d5ac63159a5dffc13e42d87e070,android,2021-01-17 17:40:24,NaN,NaN,NaT,NaN,True,False,NaN,NaN
2,a3e52e50d379c3da808c4d8864f0d996,android,2021-01-11 04:02:52,100497.0,a3e52e50d379c3da808c4d8864f0d996,2021-01-12 01:58:52,45-54,True,True,True,True
3,2f7551cdd9a0a658350394e51bc74de3,android,2021-07-22 21:00:34,109802.0,2f7551cdd9a0a658350394e51bc74de3,2021-07-24 02:41:34,Unknown,True,True,True,True
4,81adc238826a8dce8a706c083abc095e,web,2021-04-24 01:43:54,105441.0,81adc238826a8dce8a706c083abc095e,2021-04-24 01:52:54,35-44,True,True,NaN,NaN


In [ ]:
# copy the lead_base_table to a new dataframe called lead_base_table_age
lead_base_table_age = lead_base_table.copy()  

# missing values in age_range column are filled with 'Unknown'
'''try using 'df.method({col: value}, inplace=True)' 
to avoid the SettingWithCopyWarning'''
# lead_base_table_age['age_range'].fillna('Unknown', inplace=True)
lead_base_table_age.fillna({'age_range': 'Unknown'}, inplace=True)
lead_base_table_age.head()

lead_base_table_final = lead_base_table_age.copy()
print(lead_base_table_final.head())
# export the lead_base_table_final dataframe to a csv file
lead_base_table_final.to_csv('lead_base_table_final.csv', index=False)


                   app_download_key platform         download_ts   user_id  \
0  06f49bcc6895f888eba41043f95348ba  android 2021-05-13 13:12:06  106328.0   
1  60d79d5ac63159a5dffc13e42d87e070  android 2021-01-17 17:40:24       NaN   
2  a3e52e50d379c3da808c4d8864f0d996  android 2021-01-11 04:02:52  100497.0   
3  2f7551cdd9a0a658350394e51bc74de3  android 2021-07-22 21:00:34  109802.0   
4  81adc238826a8dce8a706c083abc095e      web 2021-04-24 01:43:54  105441.0   

                         session_id           signup_ts age_range  \
0  06f49bcc6895f888eba41043f95348ba 2021-05-14 05:05:06   Unknown   
1                               NaN                 NaT   Unknown   
2  a3e52e50d379c3da808c4d8864f0d996 2021-01-12 01:58:52     45-54   
3  2f7551cdd9a0a658350394e51bc74de3 2021-07-24 02:41:34   Unknown   
4  81adc238826a8dce8a706c083abc095e 2021-04-24 01:52:54     35-44   

   is_downloaded  is_signedup is_requested is_completed  
0           True         True          NaN          NaN  


In [ ]:
full_funnel_by_age = lead_base_table_age.groupby('age_range').agg({
    'is_downloaded': 'sum'
    , 'is_signedup': 'sum'
    , 'is_requested': 'count'
    , 'is_completed': 'sum' }).T

full_funnel_by_age

age_range,18-24,25-34,35-44,45-54,Unknown
is_downloaded,1865,3447,5181,1826,11289
is_signedup,1865,3447,5181,1826,5304
is_requested,1300,2425,3662,1285,3734
is_completed,670,1227,1861,630,1845


In [ ]:
# sum the values in the full_funnel_by_age dataframe by row
# axis=1 means sum by row 
print(full_funnel_by_age.sum(axis=1))
print(full_funnel)


is_downloaded    23608
is_signedup      17623
is_requested     12406
is_completed      6233
dtype: object
App Downloads       23608
Signups             17623
Ride Requests       12406
Ride Completions     6233
dtype: int64


In [ ]:
traces = list()
for age in full_funnel_by_age.columns:
  trace = Funnel(x = full_funnel_by_age[age].values,
                 y = full_funnel_by_age.index,
                 textinfo = "value+percent initial",
                 name = age)
  traces.append(trace)


fig = Figure(data = traces,
             layout = Layout(title = "Full funnel by Age Range"))
fig.show()

## Final_step: 
we will develop a dash_app where It should be easy to see the answers to business questions...